# Butterfly DDPM visualization from saved checkpoints

This notebook clones the repo from GitHub and uses a checkpoint saved from a previous training run on Google Drive.


## Mount Drive and clone repo

Point `RUN_NAME` at the run folder created by the training notebook.


In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/AAnanya19/Human-Faces-Generation-Diffusion-Models.git"
BRANCH = "main"
WORKDIR = Path("/content/Human-Faces-Generation-Diffusion-Models")

CHECKPOINTS_ROOT = Path("/content/drive/MyDrive/aml/ddpm_runs")
RUN_NAME = "butterfly_run_001"
CHECKPOINT_PATH = CHECKPOINTS_ROOT / RUN_NAME / "ddpm_final.pth"
OUTPUT_DIR = CHECKPOINTS_ROOT / RUN_NAME / "generated"

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

!git clone --branch "$BRANCH" "$REPO_URL" "$WORKDIR"

os.chdir(WORKDIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

print("Project root:", WORKDIR)
print("Checkpoint:", CHECKPOINT_PATH)
print("Output dir:", OUTPUT_DIR)


## Install dependencies


In [ ]:
assert os.path.isfile("requirements.txt"), "Run the clone cell first."

!pip install -q datasets huggingface_hub tqdm matplotlib torchvision


## Sampling config


In [ ]:
BATCH_SIZE = 8
IMAGE_SIZE = 64
TIMESTEPS = 1000


## Generate images from the saved checkpoint


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("device:", device)
if device != "cuda":
    raise RuntimeError("GPU is not active. In Colab, switch Runtime -> Change runtime type -> GPU.")

!python3 scripts/generate_butterflies.py \
    --checkpoint "$CHECKPOINT_PATH" \
    --out_dir "$OUTPUT_DIR" \
    --batch_size $BATCH_SIZE \
    --image_size $IMAGE_SIZE \
    --timesteps $TIMESTEPS \
    --device cuda


## Preview result


In [ ]:
from IPython.display import Image, display

generated_path = OUTPUT_DIR / "butterfly_grid.png"
if not generated_path.is_file():
    raise FileNotFoundError(f"Expected output not found: {generated_path}")

display(Image(filename=str(generated_path)))


To visualize a different run, change only `RUN_NAME` and re run the notebook.
